# Week 7: one income feature per account-month

**Checkpoint:** build one account-month measure from our key category (income), define the transaction filter and
amount convention, then report the account-month count, zero share, median / IQR / upper tail, one surprising
pattern, one data/label caveat, and keep **zero activity** separate from **missing coverage**.

**How this notebook uses the 5,000 memos**
1. `memos_5000.csv` (original 1,000 + 4,000 account-sampled rows) gives the income **examples** and the **accounts**.
2. The feature itself is computed on each of those accounts' **full transaction history** pulled from the parquet
   files, because the 5,000 rows only hold a few transactions per account; a month-level count from them would
   be mostly fake zeros.

Upload `memos_5000.csv` next to this notebook before running.

In [1]:
import glob, json, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.dataset as ds

BASE = "/home/ailai/teams/d12/UCSD - Data Share"
SAMPLE_CSV = "memos_5000.csv"                        # upload next to this notebook
HIST_CACHE = Path("full_histories_5000accts.parquet") # written on first run; reruns skip the slow scan
OUT = Path("outputs_income_feature"); OUT.mkdir(exist_ok=True)
FAKE = 207405
CREDIT_CODE = 1        # credit_or_debit: 1 = credit, 2 = debit (checked in section 3)
COLS = ["account_key", "posted_date", "amount", "credit_or_debit", "memo"]

## 1. The category: which credits count as income

Week 6's ordered rules (first match wins), plus three fixes found on the larger sample: `dir dep` / `PYRL`
now count as payroll, IRS tax refunds count as refunds (not benefits), and child support counts as other income. Non-income credits are matched **before** the income words
they can contain, e.g. "Cleo *salary* advance" is a cash advance, not payroll.

In [2]:
INCOME_RULES = [
    ("Not income - Cash advance / loan",
     r"\bdave\b|brigit|earnin|moneylion|instacash|cleo|empower|\btilt\b|klover|floatme|mypay|grant cash"
     r"|credit ?genie|onemain|cash advance|advance"),
    ("Not income - Refund / reversal", r"return|refund|reverse|reversal|tax ref"),
    ("Payroll", r"payroll|direct dep|dir dep|\bpyrl|payrl|gusto pay|\bsalary\b|\bpay\b ppd"),
    ("Benefits", r"\bssi\b|social security|treas 310|unemployment|\bui\b|va benefit|child benefit|province of"),
    ("Gig payout", r"uber instantpay|uber pro card|stripe|spark driver|doordash|instacart|lyft|gopuff|maplebear"),
    ("Other income", r"dividend|interest|\brent\b|airbnb|child supp"),
    ("Not income - P2P received", r"zelle|cash app|venmo|e-transfer|apple cash|visa direct|paypal"),
    ("Not income - Own-account transfer",
     r"transfer|from savings|savings|checking|wallet|round ?up|\btfr|way2save|credit builder"),
    ("Not income - Refund / reversal", r"adjustment|correction"),
    ("Deposit - source unknown", r"deposit|edeposit|\bcheck\b"),
]
INCOME_TYPES = {"Payroll", "Gig payout", "Benefits", "Other income"}
_compiled = [(lab, re.compile(p)) for lab, p in INCOME_RULES]

def income_label(text):
    for lab, rx in _compiled:
        m = rx.search(text)
        if m:
            return lab, m.group(0)
    return "Uncertain", ""

def lower_text(s):
    return re.sub(r"\s+", " ", str(s).lower()).strip()

In [3]:
sample = pd.read_csv(SAMPLE_CSV, dtype={"item_id": str, "account_key": "Int64"})
sample = sample[sample["account_key"] != FAKE]
cr = sample[sample["direction"] == "credit"].copy()
cr[["income_label", "cue"]] = cr["memo_raw"].map(lambda m: income_label(lower_text(m))).tolist()
cr["is_income"] = cr["income_label"].isin(INCOME_TYPES)

print(f"{len(sample)} sample rows, {len(cr)} credits")
(cr.groupby("income_label")
   .agg(rows=("item_id", "size"), distinct_memos=("memo_raw", "nunique"), dollars=("amount", "sum"))
   .assign(income=lambda t: t.index.isin(INCOME_TYPES))
   .sort_values("rows", ascending=False).round(0))

4959 sample rows, 1210 credits


,rows,distinct_memos,dollars,income
income_label,,,,
Not income - Own-account transfer,361,190,66680.0,False
Uncertain,217,157,38780.0,False
Not income - Cash advance / loan,160,111,21567.0,False
Not income - P2P received,156,124,22403.0,False
Deposit - source unknown,92,74,57492.0,False
Payroll,92,78,84666.0,True
Other income,66,27,5478.0,True
Not income - Refund / reversal,31,30,4378.0,False
Benefits,18,16,28989.0,True


**More examples of the category (Q1 deep dive).** Up to 6 distinct memos per label. The full list is saved to
`income_examples_review.csv` with a blank `human_label` column for the group to correct.

In [4]:
ex = (cr.drop_duplicates("memo_raw").sample(frac=1, random_state=0)
        .groupby("income_label").head(6))
pd.set_option("display.max_colwidth", 80)
display(ex[["income_label", "cue", "memo_raw", "amount"]].sort_values("income_label"))

review = (cr.groupby(["memo_raw", "income_label", "cue"], dropna=False)
            .agg(rows=("item_id", "size"), dollars=("amount", "sum")).reset_index()
            .sort_values(["income_label", "rows"], ascending=[True, False]))
review["human_label"] = ""
review.to_csv(OUT / "income_examples_review.csv", index=False)
print(len(review), "distinct credit memos ->", OUT / "income_examples_review.csv")

,income_label,cue,memo_raw,amount
1847,Benefits,treas 310,SSA TREAS 310 XXSOC SEC PPD ID: 9031736026,1305.00
706,Benefits,treas 310,Deposit - 36 TREAS 310 36 TREAS 310 DEPOSIT,19.12
2089,Benefits,treas 310,Deposit - VACP TREAS 310 VACP TREAS 310 DEPOSIT,4144.54
1290,Benefits,treas 310,ACH Deposit SSA TREAS 310 XXSOC SEC [NAME],399.00
365,Benefits,province of,Electronic Funds Transfer DEPOSIT PROVINCE OF B.C,1535.50
1744,Benefits,treas 310,"XXVA BENEF VACP TREAS 310 ID9111036002, REF*48*VA COMPENSATION *03/01/26-03/...",4109.17
3773,Deposit - source unknown,check,Check Deposit eDeposit Mobile Deposit,34.00
1015,Deposit - source unknown,deposit,deposit,91.00
3437,Deposit - source unknown,deposit,ATM CASH DEPOSIT 04/24 66 GILCREAST RD LONDONDERRY NH,44.00
2469,Deposit - source unknown,deposit,Deposit,74.00


821 distinct credit memos -> outputs_income_feature/income_examples_review.csv


## 2. Pull full histories for the sampled accounts

One pass over the 15 parquet files, keeping only these accounts. Slow the first time (similar to the first
metadata cell); the result is cached to `full_histories_5000accts.parquet`, so reruns load in seconds.
That cache holds real memo text: keep it on the hub.

In [5]:
accounts = sorted(sample["account_key"].dropna().astype(int).unique().tolist())
print(len(accounts), "accounts")

if HIST_CACHE.exists():
    hist = pd.read_parquet(HIST_CACHE)
    print("loaded cache:", len(hist), "rows")
else:
    files = sorted(glob.glob(f"{BASE}/ucsd_transactions_part_*.parquet"))
    flt = ds.field("account_key").isin(accounts)
    parts = []
    for k, f in enumerate(files, 1):
        parts.append(ds.dataset(f, format="parquet").to_table(columns=COLS, filter=flt).to_pandas())
        print(f"[{k}/{len(files)}] {sum(map(len, parts)):,} rows so far")
    hist = pd.concat(parts, ignore_index=True)
    hist["memo"] = hist["memo"].map(lambda b: b.decode("utf-8", "replace") if isinstance(b, bytes) else b)
    hist.to_parquet(HIST_CACHE, index=False)

hist = hist[hist["account_key"] != FAKE].copy()
hist["date"] = pd.to_datetime(hist["posted_date"])
hist["month"] = hist["date"].dt.to_period("M")
print(f"{len(hist):,} transactions, {hist.account_key.nunique()} accounts, "
      f"{hist.date.min().date()} to {hist.date.max().date()}")

513 accounts
[1/15] 220,845 rows so far
[2/15] 393,775 rows so far
[3/15] 482,498 rows so far
[4/15] 558,706 rows so far
[5/15] 628,722 rows so far
[6/15] 696,169 rows so far
[7/15] 778,241 rows so far
[8/15] 943,095 rows so far
[9/15] 1,165,776 rows so far
[10/15] 1,433,415 rows so far
[11/15] 1,720,044 rows so far
[12/15] 2,068,972 rows so far
[13/15] 2,485,540 rows so far
[14/15] 2,971,346 rows so far
[15/15] 3,570,107 rows so far
3,570,107 transactions, 513 accounts, 2022-07-18 to 2026-09-22


## 3. Check the direction code and amount convention

Amounts are stored positive; direction is in `credit_or_debit`. If code 1 is credits, its memos should look like
deposits / payroll / transfers in, and its typical amount should be larger.

In [6]:
print("negative amounts:", int((hist["amount"] < 0).sum()))
display(hist.groupby("credit_or_debit")["amount"].describe().round(2))
for c in sorted(hist["credit_or_debit"].unique()):
    print(f"--- code {c}:", hist.loc[hist.credit_or_debit == c, "memo"].sample(6, random_state=1).tolist())

negative amounts: 0


,count,unique,top,freq
credit_or_debit,,,,
1,901034,18920,100.00,64214
2,2669073,23865,20.00,76810


--- code 1: ['Online Transfer from SAV ...1675 transaction#: 28726195495', 'Zelle payment from [NAME] Conf# NlWPje70q', 'Albert EDI PYMNTS FCR21753369 [NAME]', 'PAYROLL UNIVERSITY OF MI 5393 PABLO QUILES JR ACH CREDIT', 'MyPay advance', 'XX9366 TRANSFER IN CASH APP KATHLEE Oakland CA ad983a51 001819']
--- code 2: ['7 Eleven', 'Tilt Advance', 'SHORTSWAVE SINGAPORE - DBT/WDR*605379100051', 'CASH APP*KOURT MACOUL*A Oakland CA 08/17', 'Point Of Sale Withdrawal WWW / COSTCO COM 800-955-2292 WAUS Card #0633', 'Purchase AMAZON MKTPLACE PMTS Amzn.com/billWAUS']


## 4. Feature definition

**Feature: `income_total` = monthly income dollars per account-month.**

- **Transaction filter:** credits (`credit_or_debit == 1`) whose memo's rule label is Payroll, Gig payout, Benefits
  or Other income. Excluded: own-account transfers, P2P received, refunds / reversals, cash advances / loans,
  deposits with no named source, and Uncertain.
- **Amount convention:** sum of the positive credit `amount`, in dollars, by calendar month of `posted_date`.
  Debits never enter the feature.
- **Unit:** account × calendar month.

In [7]:
hist["is_credit"] = hist["credit_or_debit"] == CREDIT_CODE
low = hist.loc[hist["is_credit"], "memo"].map(lower_text)
lab_by_memo = {t: income_label(t)[0] for t in low.unique()}
hist["income_label"] = None
hist.loc[hist["is_credit"], "income_label"] = low.map(lab_by_memo)
hist["is_income"] = hist["income_label"].isin(INCOME_TYPES)
hist["income_amt"] = np.where(hist["is_income"], hist["amount"], 0.0)
hist["payroll_amt"] = np.where(hist["income_label"] == "Payroll", hist["amount"], 0.0)

credits = hist[hist["is_credit"]]
lab_tab = (credits.groupby("income_label")
           .agg(rows=("amount", "size"), dollars=("amount", "sum"))
           .assign(share_of_credit_dollars=lambda t: t.dollars / t.dollars.sum())
           .sort_values("dollars", ascending=False))
display(lab_tab.round(3))

,rows,dollars,share_of_credit_dollars
income_label,,,
Not income - Own-account transfer,393889,46700359.47,0.2467785649781430549469009172
Payroll,40222,45785564.51,0.2419445168029237780571586373
Uncertain,123566,34338414.76,0.1814543788156645329432654466
Deposit - source unknown,42279,24004970.63,0.1268493919884996256954686076
Not income - Cash advance / loan,137469,13709183.48,0.07244339582417491250104902739
Not income - P2P received,131863,13155373.66,0.06951690030676463272115350646
Benefits,5432,8019344.93,0.04237659958830606124461040020
Not income - Refund / reversal,14253,2984545.13,0.01577122259127452945674500482
Gig payout,5679,304390.98,0.001608492313318175044777252079


## 5. Account-month grid: zero vs missing coverage

For each account, every calendar month from its first to its last transaction gets a row, and each row gets a
coverage status:

| status | meaning | in the denominator? |
|---|---|---|
| `covered` | full month with at least one transaction of any kind | yes, the feature can be 0 (true zero) |
| `no_data_gap` | month inside the account's span with no transactions at all | no, missing coverage, not a zero |
| `partial_edge` | the account's first or last month (observed only part of the month) | no |

A `covered` month with `income_total == 0` is **zero income activity**. A `no_data_gap` month is **missing
coverage**: we can't tell whether the person had no income or we just have no data.

In [8]:
act = (hist.groupby(["account_key", "month"])
           .agg(n_txn=("amount", "size"), n_credits=("is_credit", "sum"),
                income_total=("income_amt", "sum"), payroll_total=("payroll_amt", "sum"),
                income_count=("is_income", "sum"))
           .reset_index())

span = hist.groupby("account_key")["month"].agg(first="min", last="max").reset_index()
grid = pd.concat([pd.DataFrame({"account_key": r.account_key,
                                "month": pd.period_range(r.first, r.last, freq="M")})
                  for r in span.itertuples()], ignore_index=True)
am = (grid.merge(span, on="account_key")
          .merge(act, on=["account_key", "month"], how="left"))
for c in ["n_txn", "n_credits", "income_count"]:
    am[c] = am[c].fillna(0).astype(int)
for c in ["income_total", "payroll_total"]:
    am[c] = am[c].fillna(0.0)
am["status"] = np.select([(am.month == am["first"]) | (am.month == am["last"]), am.n_txn == 0],
                         ["partial_edge", "no_data_gap"], "covered")
am = am.drop(columns=["first", "last"])

status = am["status"].value_counts().rename("account_months").to_frame()
status["share"] = (status.account_months / len(am)).round(3)
display(status)

TypeError: unsupported operand type(s) for +: 'float' and 'decimal.Decimal'

## 6. Distribution of `income_total` over covered account-months

In [ ]:
cov = am[am["status"] == "covered"]
x = cov["income_total"]
nz = x[x > 0]
q = x.quantile([.25, .5, .75, .9, .95, .99])
summary = {
    "denominator_account_months": int(len(cov)),
    "accounts": int(cov.account_key.nunique()),
    "zero_share": float((x == 0).mean()),
    "median": float(q[.5]), "p25": float(q[.25]), "p75": float(q[.75]), "iqr": float(q[.75] - q[.25]),
    "p90": float(q[.9]), "p95": float(q[.95]), "p99": float(q[.99]), "max": float(x.max()),
    "nonzero_median": float(nz.median()) if len(nz) else None,
    "nonzero_p25": float(nz.quantile(.25)) if len(nz) else None,
    "nonzero_p75": float(nz.quantile(.75)) if len(nz) else None,
    "excluded_no_data_gap": int((am.status == "no_data_gap").sum()),
    "excluded_partial_edge": int((am.status == "partial_edge").sum()),
}
pd.Series(summary).to_frame("value")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1, 2]})

parts = pd.Series({
    "income\n> 0": int((x > 0).sum()),
    "income\n= 0": int((x == 0).sum()),
    "missing\n(no data)": summary["excluded_no_data_gap"],
    "partial\nmonth": summary["excluded_partial_edge"],
})
axes[0].bar(parts.index, parts.values, color=["#2a7fba", "#9cc3e0", "#c9c9c9", "#e2e2e2"])
axes[0].set_ylabel("account-months")
axes[0].set_title("Zero income vs missing coverage")
axes[0].tick_params(axis="x", labelsize=8)
for i, v in enumerate(parts.values):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

if len(nz):
    bins = np.logspace(np.log10(max(nz.min(), 1)), np.log10(nz.max()), 40)
    axes[1].hist(nz, bins=bins, color="#2a7fba")
    axes[1].set_xscale("log")
    for name, v in [("median", nz.median()), ("p75", nz.quantile(.75)), ("p95", nz.quantile(.95))]:
        axes[1].axvline(v, color="k", ls="--", lw=0.8)
        axes[1].text(v, axes[1].get_ylim()[1] * 0.95, f" {name} ${v:,.0f}", fontsize=8, va="top")
axes[1].set_xlabel("monthly income dollars (log scale)")
axes[1].set_ylabel("covered account-months")
axes[1].set_title(f"Non-zero income_total (zero share = {summary['zero_share']:.0%} not shown)")

plt.tight_layout()
plt.savefig(OUT / "income_total_distribution.png", dpi=150)
plt.show()

## 7. Material for the "surprising pattern" and the caveat

These cells only compute candidates; pick one pattern and one caveat for the write-up after reading them.

In [ ]:
# (a) how much of the money coming in is actually income?
cov_keys = set(map(tuple, cov[["account_key", "month"]].astype(str).values))
cc = credits[[k in cov_keys for k in zip(credits.account_key.astype(str), credits.month.astype(str))]]
print(f"income share of credit dollars in covered months: {cc.loc[cc.is_income, 'amount'].sum() / cc.amount.sum():.1%}")

# (b) accounts that never show income in any covered month
per_acct = cov.groupby("account_key").agg(months=("month", "size"),
                                          zero_months=("income_total", lambda s: (s == 0).sum()))
never = (per_acct.zero_months == per_acct.months)
print(f"accounts with zero income in every covered month: {never.sum()} of {len(per_acct)} ({never.mean():.0%})")

# (c) payroll vs other income inside the feature
print(f"payroll share of income dollars: {cov.payroll_total.sum() / cov.income_total.sum():.1%}")

# (d) upper tail: the biggest account-months and what drove them
top = cov.nlargest(10, "income_total")[["account_key", "month", "income_total", "payroll_total", "income_count"]]
display(top)
top_keys = set(zip(top.account_key, top.month.astype(str)))
in_top = pd.Series([(a, str(m)) in top_keys for a, m in zip(credits.account_key, credits.month)],
                   index=credits.index)
drivers = credits[in_top & credits.is_income]
display(drivers.groupby(["income_label", "memo"])["amount"].agg(["size", "sum"])
               .sort_values("sum", ascending=False).head(15))

In [ ]:
# label caveat: big credit dollars the rules could not place, which may hide income
unk = credits[credits.income_label.isin(["Uncertain", "Deposit - source unknown"])].copy()
unk["memo_pattern"] = (unk["memo"].map(lower_text)
                       .str.replace(r"\d+", "#", regex=True).str.replace(r"\s+", " ", regex=True).str[:60])
print(f"credit dollars labeled Uncertain or source-unknown deposit: "
      f"{unk.amount.sum() / credits.amount.sum():.1%} of all credit dollars")
display(unk.groupby(["income_label", "memo_pattern"])["amount"].agg(rows="size", dollars="sum")
           .sort_values("dollars", ascending=False).head(20).round(0))

## 8. Save outputs

In [ ]:
am.assign(month=am.month.astype(str)).to_csv(OUT / "account_month_income.csv", index=False)
(OUT / "income_feature_summary.json").write_text(json.dumps(summary, indent=2))
print(sorted(p.name for p in OUT.iterdir()))

## 9. Write-up (fill in from the numbers above)

- **Feature formula:** `income_total` = sum of credit amounts per account-month whose memo is labeled Payroll,
  Gig payout, Benefits or Other income by the ordered income rules.
- **Denominator:** ___ covered account-months from ___ accounts (excluded: ___ no-data gap months,
  ___ partial edge months).
- **Distribution:** zero share ___%; median $___; IQR $___ to $___; p95 $___; p99 $___.
- **Chart:** `outputs_income_feature/income_total_distribution.png`
- **Surprising pattern:** ___
- **Caveat:** ___ (e.g. labels are rule suggestions, not human-reviewed; accounts were sampled by activity, so
  heavy users are over-represented; source-unknown deposits might be cash income)